## 00 — Índice de Documentação

## Descrição

Este documento serve como ponto de entrada para toda a documentação técnica do projeto **Engenharia de Dados - Projeto Final**. O repositório implementa um pipeline completo de dados de e-commerce, desde a origem NoSQL (MongoDB) até um modelo dimensional analítico (camada Gold), utilizando a arquitetura medalhão sobre um Data Lake com orquestração Apache Airflow.

A documentação está organizada em 10 notebooks autocontidos, projetados para permitir que desenvolvedores, engenheiros de dados, arquitetos e stakeholders compreendam o sistema de forma progressiva — desde uma visão de negócio até a execução prática do pipeline.

---

## Objetivos deste Índice

- Apresentar a estrutura e ordem recomendada de leitura da documentação
- Fornecer um mapa visual do sistema e seus componentes principais
- Definir termos técnicos no glossário para padronização da comunicação
- Servir como referência rápida para localizar informações específicas

---

## Estrutura da Documentação

| Ordem | Notebook | Descrição |
|:-----:|----------|-------------|
| 01 | `00_indice_documentacao.ipynb` | **Você está aqui** — Índice mestre, glossário e mapa do sistema |
| 02 | `01_visao_geral_projeto.ipynb` | Contexto de negócio, problemas resolvidos, stack tecnológico e arquitetura de alto nível |
| 03 | `02_estrutura_repositorio.ipynb` | Organização das pastas, módulos Python, responsabilidades de cada diretório |
| 04 | `03_arquitetura_detalhada.ipynb` | Deep dive na arquitetura medalhão, Delta Lake, SCD Tipo 2, padrões de design e fluxos de execução |
| 05 | `04_processos_negocio.ipynb` | Processos de negócio: geração de dados, carga incremental, deduplicação, validação de qualidade e modelagem dimensional |
| 06 | `05_fluxo_dados.ipynb`  | Fluxo ponta a ponta do dado com diagramas de sequência detalhados de cada DAG |
| 07 | `06_banco_dados.ipynb`  | Modelagem do MongoDB: coleções, schemas `$jsonSchema`, relacionamentos, índices e estratégia de incrementalidade |
| 08 | `07_interfaces_apis.ipynb` | Contratos internos entre DAGs e jobs Spark, manifestos JSON, conexões Airflow e argumentos de CLI |
| 09 | `08_infraestrutura.ipynb`  | Docker Compose, stack de serviços, CI/CD GitHub Actions, MkDocs e observabilidade |
| 10 | `09_execucao_pipeline.ipynb` | Guia prático passo a passo: do `.env` à execução completa do pipeline e validação dos resultados |

---

## Mapa do Sistema

```mermaid
graph TD
    A[MongoDB/Atlas<br/>10 coleções] -->|DAG: mongodb_to_landing| B[Landing<br/>JSON bruto]
    B -->|DAG: landing_to_bronze| C[Bronze<br/>Delta Lake]
    C -->|DAG: bronze_to_silver| D[Silver<br/>Delta Limpo]
    D -->|DAG: silver_to_gold| E[Gold<br/>Modelo Dimensional]
    E -->|Consumo| F[Dashboard / BI]

    subgraph Orquestração
        G[Apache Airflow<br/>4 DAGs]
    end

    subgraph Data Lake
        B
        C
        D
        E
    end

    G -.->|Agendamento| A
    G -.->|Execução| B
    G -.->|Execução| C
    G -.->|Execução| D
    G -.->|Execução| E

    style A fill:#15803d,color:#fff,stroke:#166534
    style B fill:#e2e8f0,stroke:#94a3b8,color:#1e293b
    style C fill:#fdba74,stroke:#b45309,color:#7c2d12
    style D fill:#cbd5e1,stroke:#64748b,color:#1e293b
    style E fill:#fde047,stroke:#a16207,color:#713f12
    style F fill:#ede9fe,stroke:#7c3aed,color:#4c1d95
    style G fill:#bfdbfe,stroke:#3b82f6,color:#1e3a8a
```

---

## Glossário

| Termo | Definição | Contexto no Projeto |
|-------|-----------|---------------------|
| **Medalhão** | Arquitetura de Data Lake em camadas: Landing, Bronze, Silver, Gold | Organização do Data Lake em 4 camadas com responsabilidades distintas |
| **Landing** | Camada de ingestão bruta, cópia fiel da origem sem transformação | Recebe JSON estendido do MongoDB, preservando tipos BSON |
| **Bronze** | Camada de persistência em formato analítico (Delta Lake) com metadados de auditoria | JSON → Delta Lake, com colunas de controle (`_bronze_ingested_at`, etc.) |
| **Silver** | Camada de limpeza, tipagem, deduplicação e validação de qualidade | Aplica regras de negócio, integridade referencial, rejeita registros inválidos |
| **Gold** | Camada de modelagem dimensional para consumo analítico (BI) | 4 dimensões (SCD Tipo 2) + 4 fatos, pronto para dashboard |
| **SCD Tipo 2** | Slowly Changing Dimension Type 2 — preserva histórico de alterações de atributos | Dimensões `dim_cliente`, `dim_produto`, `dim_cupom` versionam atributos ao longo do tempo |
| **Delta Lake** | Format de storage com transações ACID, time-travel e MERGE incremental | Usado nas camadas Bronze, Silver e Gold sobre MinIO/S3 |
| **DAG** | Directed Acyclic Graph — grafo direcionado acíclico de tarefas no Airflow | Cada pipeline (mongodb→landing, landing→bronze, etc.) é uma DAG |
| **MERGE** | Operação SQL que combina INSERT e UPDATE em uma única transação | Usado para carga incremental idempotente nas camadas Silver e Gold |
| **JSON Estendido** | MongoDB Extended JSON — representação canonical de tipos BSON em JSON | Permite preservar tipos (ISODate, NumberInt, etc.) no formato texto |
| **Checkpoint** | Marcador de progresso que registra o último ponto processado | Airflow Variables guardam o maior `updated_at` processado por coleção |
| **Idempotência** | Propriedade de uma operação que pode ser executada múltiplas vezes sem efeitos colaterais | Cada execução de DAG processa apenas dados novos ou alterados |
| **Surrogate Key** | Chave artificial única substituindo a chave natural em dimensões | `cliente_sk`, `produto_sk` geradas por hash SHA256 na Gold |
| **Natural Key** | Chave de negócio original, estável entre versões | `cliente_key` = `id_cliente` na camada Gold |
| **Integridade Referencial** | Garantia que chaves estrangeiras apontam para registros existentes | Validada na Silver: registros órfãos são rejeitados para log de qualidade |
| **Quality Log** | Tabela Delta append-only que registra registros rejeitados por regras de qualidade | `silver/_control/quality_log/` documenta violações com metadados |
| **Manifesto** | Arquivo JSON de auditoria gerado por execução de DAG | Documenta totais, status e métricas de cada execução no S3 |
| **MinIO** | Object storage compatível com S3, usado como Data Lake local | Substituto local do Amazon S3 para desenvolvimento e testes |
| **SparkSubmitOperator** | Operador Airflow que submete jobs Apache Spark | Usado nas DAGs landing→bronze, bronze→silver, silver→gold |
| **PySpark** | API Python para Apache Spark | Jobs de transformação entre camadas (JSON→Delta, limpeza, modelagem) |
| **Object Storage** | Sistema de armazenamento de objetos (S3, MinIO) para Data Lake | `datalake` bucket com prefixos por camada e tabela |
| **Partitioning** | Divisão física dos dados por coluna (ex: `ano`) para otimização | Fatos na Gold são particionados por `ano` para queries eficientes |
| **Time-Travel** | Capacidade de consultar versões históricas dos dados no Delta Lake | Suporte nativo do Delta Lake para auditoria e rollback |
| **Schema Enforcement** | Rejeição de escritas que não conformam com o schema definido | Delta Lake valida schema nas camadas Bronze, Silver, Gold |

---

## Achados Principais

1. **Arquitetura Completa**: O projeto implementa um pipeline de dados realista, desde uma origem NoSQL (MongoDB) até um modelo dimensional Kimball pronto para BI.
2. **Medalhão Conciso**: Cada camada (Landing, Bronze, Silver, Gold) tem responsabilidades claras e contratos versionados em JSON.
3. **Orquestração Profissional**: Apache Airflow gerencia 4 DAGs com dependências, agendamento, retentativa e variáveis de ambiente.
4. **Qualidade de Dados**: A camada Silver implementa deduplicação, validação de domínio, integridade referencial e log de rejeições.
5. **Histórico Versionado**: Dimensões SCD Tipo 2 na Gold preservam todas as versões de atributos ao longo do tempo.
6. **Infraestrutura Containerizada**: Stack completo em Docker Compose (MongoDB, MinIO, Postgres, Airflow) para desenvolvimento local.
7. **Testes Automatizados**: Testes unitários com mocks de S3, testes de integração com PySpark/Delta e CI/CD via GitHub Actions.
8. **Documentação Publicada**: MkDocs gera site estático com Material Design, publicado automaticamente no GitHub Pages.

---

## Recomendações

1. **Execute o notebook `09_execucao_pipeline.ipynb`** em um ambiente com Docker para validar a teoria na prática.
2. **Consulte o glossário** sempre que encontrar termos técnicos desconhecidos.
3. **Mantenha este índice atualizado** ao adicionar novos notebooks ou alterar a estrutura da documentação.

---

## Conclusão

Este índice documenta um sistema de engenharia de dados de nível profissional, com arquitetura medalhão, orquestração Airflow, processamento Spark/Delta Lake e modelagem dimensional. A documentação foi projetada para ser progressiva, autocontida e prática, executem e mantenham o sistema.
